# Populate Configurable-Greedy Generations

Construct one fixed random population, search every member, archive the best descendants, and feed exactly those descendants into the next generation. This preserves independent ancestry lines while allowing repeated passes through search basins.

In [1]:
# Experiment Configuration

RANDOM_SEED = 202_608_088
N_VERTICES = 43

# X independent ancestry lines and G descendant generations.
POPULATION_SIZE = 50
GENERATIONS = 10

# 0.00 = smallest positive exact-score reduction.
# 1.00 = ordinary maximum exact-score reduction.
GREEDINESS = 0.00

# Continue through positive-reward walls using the least-damaging
# available move. With tabu this makes repeated generations meaningful.
FALLBACK_AT_WALL = True
SEARCH_STEPS_PER_GENERATION = 500
EDGE_TABU_TENURE = 20
VISITED_STATE_WINDOW = 2_000

REPORT_EVERY_MEMBERS = 10
RUN_NAME = "configurable-greedy-generations-g000-001"
DATABASE_FILENAME = "configurable_greedy_generations.sqlite3"

In [2]:
# Imports and Project Setup

from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

from ramsey import (
    RConfigurableGreedyPolicy,
    RConfigurableGreedyPolicyConfig,
    REnvironment,
    REnvironmentConfig,
    RGenerationalExperiment,
    RGenerationalExperimentConfig,
    RGraph,
    RMonochromaticObjective,
    RProblem,
    RRandomConstruction,
    RSearch,
    RSQLiteArchive,
    RTabuMemory,
    RTabuMemoryConfig,
)

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "ramsey").is_dir():
    raise RuntimeError(
        "Run this notebook from the RamseyNumber root "
        "or notebooks directory."
    )

DATABASE_PATH = (
    project_root
    / "data"
    / "experiments"
    / DATABASE_FILENAME
)

In [3]:
# Assemble Graph, Search, Archive, and Generation Runner

construction_rng = np.random.default_rng(RANDOM_SEED)
action_rng = np.random.default_rng(RANDOM_SEED + 1)

graph = RGraph(RProblem.r55(n_vertices=N_VERTICES))
construction = RRandomConstruction(construction_rng)

memory = RTabuMemory(
    graph.number_of_edges,
    RTabuMemoryConfig(
        edge_tenure=EDGE_TABU_TENURE,
        visited_state_window=VISITED_STATE_WINDOW,
    ),
)
environment = REnvironment(
    graph=graph,
    objective=RMonochromaticObjective(),
    memory=memory,
    config=REnvironmentConfig(
        max_steps=SEARCH_STEPS_PER_GENERATION,
        use_aspiration=True,
    ),
)
policy = RConfigurableGreedyPolicy(
    rng=action_rng,
    config=RConfigurableGreedyPolicyConfig(
        greediness=GREEDINESS,
        fallback_at_wall=FALLBACK_AT_WALL,
    ),
)
search = RSearch(environment, policy)

existing_archive = globals().get("archive")
if existing_archive is not None:
    existing_archive.close()

archive = RSQLiteArchive(DATABASE_PATH)
experiment = RGenerationalExperiment(
    graph=graph,
    initial_construction=construction,
    search=search,
    archive=archive,
)
experiment_config = RGenerationalExperimentConfig(
    run_name=RUN_NAME,
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    record_steps=False,
    stop_on_solution=True,
)

print("Edges:", f"{graph.number_of_edges:,}")
print("K5s:", f"{graph.subgraph_index(5).clique_count:,}")
print("Population:", POPULATION_SIZE)
print("Generations:", GENERATIONS)
print("Greediness:", f"{GREEDINESS:.2f}")
print("Fallback at wall:", FALLBACK_AT_WALL)
print("Steps per generation:", SEARCH_STEPS_PER_GENERATION)
print("Database:", DATABASE_PATH.resolve())

TypeError: RConfigurableGreedyPolicyConfig.__init__() got an unexpected keyword argument 'fallback_at_wall'

In [ ]:
# Progress Observer

progress = {
    "start": perf_counter(),
    "best": archive.best_score(graph),
}

def report_member(member_result):
    result = member_result.search_result
    member_number = member_result.member + 1

    if member_result.new_archive_best:
        progress["best"] = result.best_score
        print(
            f"NEW RECORD: {result.best_score} | "
            f"generation={member_result.generation} | "
            f"member={member_result.member} | "
            f"archive_id={member_result.archive_record.coloring_id}"
        )

    if (
        member_number % REPORT_EVERY_MEMBERS != 0
        and member_number != POPULATION_SIZE
    ):
        return

    elapsed = perf_counter() - progress["start"]
    print(
        f"Generation {member_result.generation:2d} | "
        f"{member_number:3d}/{POPULATION_SIZE:3d} | "
        f"initial={result.initial_score:4d} | "
        f"best={result.best_score:4d} | "
        f"steps={result.steps_completed:4d} | "
        f"archive_best={progress['best']} | "
        f"elapsed={elapsed:8.1f}s"
    )

In [ ]:
# Run All Generations

experiment_start = perf_counter()

generation_result = experiment.run(
    experiment_config,
    observer=report_member,
)

experiment_elapsed = perf_counter() - experiment_start

print()
print("Generations completed:", generation_result.generations_completed)
print("Best score:", generation_result.best_score)
print("Archive best:", archive.best_score(graph))
print("Elapsed:", f"{experiment_elapsed:.1f}s")

In [ ]:
# Generation Report

report_rows = []

for generation in generation_result.generations:
    scores = np.asarray(
        [member.search_result.best_score for member in generation.members],
        dtype=np.int32,
    )
    reductions = np.asarray(
        [
            member.search_result.best_score_reduction
            for member in generation.members
        ],
        dtype=np.int32,
    )
    report_rows.append({
        "generation": generation.generation,
        "mean_initial": generation.mean_initial_score,
        "mean_child": generation.mean_child_score,
        "median_child": float(np.median(scores)),
        "minimum_child": int(scores.min()),
        "mean_reduction": float(reductions.mean()),
        "improved": int(np.count_nonzero(reductions > 0)),
        "mean_steps": generation.mean_steps,
        "exhausted": sum(
            member.search_result.exhausted for member in generation.members
        ),
    })

generation_report = pd.DataFrame(report_rows)
display(generation_report.round(2))

print()
print("Best score found:", generation_result.best_score)
print("Persistent archive best:", archive.best_score(graph))

In [ ]:
# Release SQLite

archive.close()
print("Archive closed.")